In [1]:
# We will outline jokes and then explain them.
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
import os

In [2]:
API_KEY = os.environ.get("GEMINI_API_KEY")
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.2,
    google_api_key=API_KEY)

In [3]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState) -> str:
    prompt = f"Generate a joke about {state['topic']}."
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def explain_joke(state: JokeState) -> str:
    prompt = f"Explain the joke: {state['joke']}."
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)
graph.add_node('generate_joke', generate_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke(
    {'topic': 'cats'}, config=config1
)

{'topic': 'cats',
 'joke': 'Why was the cat sitting on the computer?  To keep an eye on the mouse!',
 'explanation': 'The joke plays on the double meaning of "mouse."\n\n* **Mouse (computer):**  A small device used to control a computer.\n* **Mouse (animal):** A small rodent.\n\nThe humor comes from the unexpected shift in meaning.  We initially think of the computer mouse, but the punchline reveals the cat is watching for a real mouse.  It\'s a simple pun that relies on wordplay.'}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'cats', 'joke': 'Why was the cat sitting on the computer?  To keep an eye on the mouse!', 'explanation': 'The joke plays on the double meaning of "mouse."\n\n* **Mouse (computer):**  A small device used to control a computer.\n* **Mouse (animal):** A small rodent.\n\nThe humor comes from the unexpected shift in meaning.  We initially think of the computer mouse, but the punchline reveals the cat is watching for a real mouse.  It\'s a simple pun that relies on wordplay.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0734be-df12-6ee1-8002-ce2516e933a4'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-08-07T05:04:00.023318+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0734be-d25d-6b20-8001-c97a5f28a76f'}}, tasks=(), interrupts=())

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cats', 'joke': 'Why was the cat sitting on the computer?  To keep an eye on the mouse!', 'explanation': 'The joke plays on the double meaning of "mouse."\n\n* **Mouse (computer):**  A small device used to control a computer.\n* **Mouse (animal):** A small rodent.\n\nThe humor comes from the unexpected shift in meaning.  We initially think of the computer mouse, but the punchline reveals the cat is watching for a real mouse.  It\'s a simple pun that relies on wordplay.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0734be-df12-6ee1-8002-ce2516e933a4'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-08-07T05:04:00.023318+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0734be-d25d-6b20-8001-c97a5f28a76f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'cats', 'joke': 'Why was the cat sitting on the com

In [13]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke(
    {'topic': 'dogs'}, config=config2
)

{'topic': 'dogs',
 'joke': 'Why are dogs such bad dancers?  Because they have two left feet... and two right feet!',
 'explanation': 'The humor lies in the absurdity and unexpectedness.  The setup creates an expectation of a typical reason why a dog might be a bad dancer (e.g., lack of coordination, clumsy movements).  The punchline subverts this expectation by playing on the literal meaning of "two left feet."  Dogs, of course, don\'t have left and right feet in the human sense; the joke highlights this incongruity for a humorous effect.  It\'s a silly, nonsensical punchline that relies on wordplay.'}

In [14]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'dogs', 'joke': 'Why are dogs such bad dancers?  Because they have two left feet... and two right feet!', 'explanation': 'The humor lies in the absurdity and unexpectedness.  The setup creates an expectation of a typical reason why a dog might be a bad dancer (e.g., lack of coordination, clumsy movements).  The punchline subverts this expectation by playing on the literal meaning of "two left feet."  Dogs, of course, don\'t have left and right feet in the human sense; the joke highlights this incongruity for a humorous effect.  It\'s a silly, nonsensical punchline that relies on wordplay.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0734f8-4150-6335-800d-4cd49d5b633c'}}, metadata={'source': 'loop', 'step': 13, 'parents': {}, 'thread_id': '2'}, created_at='2025-08-07T05:29:40.406558+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0734f8-36b9-6a36-800c-524

In [19]:
workflow.get_state({'configurable': {'thread_id': '1', 'checkpoint_id': '1f0734be-cb56-6075-8000-186b154e4c5b'}})

StateSnapshot(values={'topic': 'cats'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f0734be-cb56-6075-8000-186b154e4c5b'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}, 'thread_id': '1'}, created_at='2025-08-07T05:03:57.953650+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0734be-cb4f-6ea1-bfff-e2cc577aeaa9'}}, tasks=(PregelTask(id='cab679e1-e9e5-5f4e-fc3c-eb788a0bc872', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why was the cat sitting on the computer?  To keep an eye on the mouse!'}),), interrupts=())

In [18]:
workflow.invoke(None, config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f0734be-cb56-6075-8000-186b154e4c5b'}})

{'topic': 'cats',
 'joke': 'Why was the cat sitting on the computer?  To keep an eye on the mouse!',
 'explanation': 'The humor lies in the double meaning of "mouse."  \n\n* **Mouse (computer):**  A small device used to control a computer.\n* **Mouse (animal):** A small rodent.\n\nThe joke plays on the listener\'s expectation that the cat is sitting on the computer for a reason related to computers.  Instead, the punchline reveals the cat\'s motivation is to hunt a real mouse, creating a surprising and slightly silly contrast.'}